# 3 · Probabilités et mesure de la qualité — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch03_probabilites/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « pourquoi ? » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch03_probabilites/03_notebook.ipynb`, créée par `python tools/start_chapter.py 3`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 3.1, 3.2, 3.3, 3.5, 3.6, 3.7 | vérifier tes exercices papier ✏️ | ✏️ | ★ à ★★ | |
| A | 3.12 | Dix mille fléchettes : estimer des aires (et π) | 🔬 | ★ | 15 |
|  | 3.13 | Deux disques : P(A\|B) = P(B\|A) ? | 🔮 | ★ | 10 |
|  | 3.14 | Penguins : espèce × île avec pd.crosstab | 📦 | ★★ | 20 |
| B | 3.15 | confusion_matrix à la manière de scikit-learn | 🔨 | ★★ | 20 |
|  | 3.16 | accuracy, precision, recall, F-beta et F1 (cas binaire) | 🔨 | ★★★ | 40 |
|  | 3.17 | La matrice à l'envers | 🐛 | ★★ | 15 |
|  | 3.18 | Tout positif, un seul positif : prédire les scores | 🔮 | ★★ | 15 |
|  | 3.19 | Le tableau de bord complet : classification_rates | 🔨 | ★★ | 20 |
| C | 3.20 | Un seuil sur la nageoire : precision et recall en balance | 🔬 | ★★ | 25 |
|  | 3.21 | Simuler le dépistage : la prévalence fait la precision | 🔬 | ★★ | 30 |
|  | 3.22 | Vérifier avec scikit-learn : classification_report et affichages | 📦 | ★★ | 20 |
|  | 3.23 | Lire la documentation de sklearn.metrics | 🛠️ | ★★ | 25 |
| D | 3.24 | Courbe ROC et AUC | 🔨 | ★★★ | 40 |
|  | 3.25 | Moyennes macro, micro et pondérée | 🔨 | ★★★ | 35 |
|  | 3.26 | Courbe precision-recall et average precision | 🔨 | ★★★ | 40 |
|  | 3.27 | ROC ou PR ? Lire les courbes d'un problème déséquilibré | 📈 | ★★ | 25 |
|  | 3.28 | Calibration : quand la météo annonce 70 % | 🔨 | ★★★ | 35 |
|  | 3.29 | Recall ≥ 0,99 au meilleur prix | 🏆 | ★★★ | 45 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="3")  # your own library

## Objectifs et rappel express

- Calculer des probabilités simples, conditionnelles, jointes et marginales, et ne pas confondre $P(A \mid B)$ avec $P(B \mid A)$.
- Construire une matrice de confusion et en tirer accuracy, precision, recall, F1 et les autres mesures, avec les conventions de scikit-learn.
- Choisir une mesure et un seuil selon le coût des erreurs et la prévalence.
- Évaluer des scores sur tous les seuils (courbes ROC et precision-recall) et vérifier des probabilités (calibration).

**Rappel express.** $P(A \mid B) = \frac{P(A, B)}{P(B)}$ ; $P(A, B) = P(A \mid B)\,P(B)$ ; matrice de confusion de scikit-learn pour des étiquettes 0/1 : `[[TN, FP], [FN, TP]]` ; accuracy $= \frac{TP + TN}{n}$ ; precision $= \frac{TP}{TP + FP}$ ; recall $= \frac{TP}{TP + FN}$ ; spécificité $= \frac{TN}{TN + FP}$ ; $F_1 = \frac{2\,TP}{2\,TP + FP + FN}$. En Python : `pd.crosstab`, `sklearn.metrics.confusion_matrix`, `classification_report`, `roc_curve`, `precision_recall_curve`.

## Partie 0 · Vérifier tes exercices papier (✏️ 3.1, 3.2, 3.3, 3.5, 3.6 et 3.7)

Fais d'abord les exercices ✏️ de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée (par exemple `42`, `0.125`, `[1, 2, 3]` ou `True`), pas l'expression Python, sinon tu ne vérifies rien. Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les exercices ∂ 3.4 et 3.8 se corrigent avec `05_solutions.md`.

**Ex 3.1 — Fléchettes et aires : probabilités simples et conditionnelles**

In [ ]:
# Ex 3.1: the values you found on paper
answer_3_1a = ...  # a) P(A)
answer_3_1b = ...  # b) P(B)
answer_3_1c = ...  # c) P(A, B)
answer_3_1d = ...  # d) P(A | B)
answer_3_1e = ...  # e) P(B | A), 3 decimals
answer_3_1f = ...  # f) P(neither A nor B)
answer_3_1g = ...  # g) expected darts in B out of 400
answer_3_1h = ...  # h) expected darts in A and B out of 400
answer_3_1i = ...  # i) the student's estimate of P(A | B), 3 decimals
answer_3_1j = ...  # j) are A and B independent? True or False

for letter, answer in zip("abcdefghij", [answer_3_1a, answer_3_1b, answer_3_1c, answer_3_1d, answer_3_1e, answer_3_1f, answer_3_1g, answer_3_1h, answer_3_1i, answer_3_1j]):
    wb.check(f"3.1{letter}", answer)

**Ex 3.2 — Les 20 points : matrice de confusion et quatre mesures**

In [ ]:
# Ex 3.2: the values you found on paper
answer_3_2a = ...  # a) TP
answer_3_2b = ...  # b) FP
answer_3_2c = ...  # c) FN
answer_3_2d = ...  # d) TN
answer_3_2e = ...  # e) the confusion matrix as scikit-learn returns it, a list of 2 rows
answer_3_2f = ...  # f) the accuracy, 2 decimals
answer_3_2g = ...  # g) the precision, 3 decimals
answer_3_2h = ...  # h) the recall, 2 decimals
answer_3_2i = ...  # i) the F1 score, 3 decimals

for letter, answer in zip("abcdefghi", [answer_3_2a, answer_3_2b, answer_3_2c, answer_3_2d, answer_3_2e, answer_3_2f, answer_3_2g, answer_3_2h, answer_3_2i]):
    wb.check(f"3.2{letter}", answer)

**Ex 3.3 — Le glacier : jointes, marginales et conditionnelles**

In [ ]:
# Ex 3.3: the values you found on paper
answer_3_3a = ...  # a) P(V)
answer_3_3b = ...  # b) P(C), 3 decimals
answer_3_3c = ...  # c) P(V, C)
answer_3_3d = ...  # d) P(V | C), 3 decimals
answer_3_3e = ...  # e) P(C | V)
answer_3_3f = ...  # f) P(chocolate | cup), 3 decimals
answer_3_3g = ...  # g) are the flavour and the container independent? True or False
answer_3_3h = ...  # h) vanilla cones to plan for 300 customers

for letter, answer in zip("abcdefgh", [answer_3_3a, answer_3_3b, answer_3_3c, answer_3_3d, answer_3_3e, answer_3_3f, answer_3_3g, answer_3_3h]):
    wb.check(f"3.3{letter}", answer)

**Ex 3.5 — Toutes les mesures du tableau récapitulatif**

In [ ]:
# Ex 3.5: the values you found on paper
answer_3_5a = ...  # a) the prevalence, 3 decimals
answer_3_5b = ...  # b) the accuracy, 3 decimals
answer_3_5c = ...  # c) the precision, 3 decimals
answer_3_5d = ...  # d) the recall, 3 decimals
answer_3_5e = ...  # e) the specificity, 3 decimals
answer_3_5f = ...  # f) the NPV, 3 decimals
answer_3_5g = ...  # g) the FPR, 3 decimals
answer_3_5h = ...  # h) the FNR, 3 decimals
answer_3_5i = ...  # i) the FDR, 3 decimals
answer_3_5j = ...  # j) the FOR (false omission rate), 3 decimals
answer_3_5k = ...  # k) the F1 score, 3 decimals
answer_3_5l = ...  # l) the balanced accuracy, 3 decimals
answer_3_5m = ...  # m) the MCC, 3 decimals

for letter, answer in zip("abcdefghijklm", [answer_3_5a, answer_3_5b, answer_3_5c, answer_3_5d, answer_3_5e, answer_3_5f, answer_3_5g, answer_3_5h, answer_3_5i, answer_3_5j, answer_3_5k, answer_3_5l, answer_3_5m]):
    wb.check(f"3.5{letter}", answer)

**Ex 3.6 — Trois espèces : moyennes macro, micro et pondérée**

In [ ]:
# Ex 3.6: the values you found on paper
answer_3_6a = ...  # a) the support of each species, a list
answer_3_6b = ...  # b) the precision of each species, a list (3 decimals)
answer_3_6c = ...  # c) the recall of each species, a list (3 decimals)
answer_3_6d = ...  # d) the F1 score of each species, a list (3 decimals)
answer_3_6e = ...  # e) the macro precision, 3 decimals
answer_3_6f = ...  # f) the macro F1 score, 3 decimals
answer_3_6g = ...  # g) the weighted F1 score, 3 decimals
answer_3_6h = ...  # h) the micro F1 score, 3 decimals

for letter, answer in zip("abcdefgh", [answer_3_6a, answer_3_6b, answer_3_6c, answer_3_6d, answer_3_6e, answer_3_6f, answer_3_6g, answer_3_6h]):
    wb.check(f"3.6{letter}", answer)

**Ex 3.7 — Le test « fiable à 99 % » dans une ville à 1 % de malades**

In [ ]:
# Ex 3.7: the values you found on paper
answer_3_7a = ...  # a) the number of sick people
answer_3_7b = ...  # b) TP
answer_3_7c = ...  # c) FN
answer_3_7d = ...  # d) FP
answer_3_7e = ...  # e) TN
answer_3_7f = ...  # f) the precision P(sick | positive), 2 decimals
answer_3_7g = ...  # g) the NPV P(healthy | negative), 4 decimals
answer_3_7h = ...  # h) the accuracy, 2 decimals
answer_3_7i = ...  # i) the precision with a prevalence of 0.2 %, 3 decimals
answer_3_7j = ...  # j) the precision with a prevalence of 10 %, 3 decimals
answer_3_7k = ...  # k) the book's specificity TN / (TN + FP), 2 decimals
answer_3_7l = ...  # l) the book's NPV TN / (TN + FN), 4 decimals

for letter, answer in zip("abcdefghijkl", [answer_3_7a, answer_3_7b, answer_3_7c, answer_3_7d, answer_3_7e, answer_3_7f, answer_3_7g, answer_3_7h, answer_3_7i, answer_3_7j, answer_3_7k, answer_3_7l]):
    wb.check(f"3.7{letter}", answer)

## Partie A · Probabilités par les aires et par les tables

Fiche §3.2 à §3.6. Tu estimes une aire en lançant des fléchettes simulées, tu vérifies quand $P(A \mid B)$ et $P(B \mid A)$ diffèrent, puis tu lis des probabilités jointes, marginales et conditionnelles dans une table de contingence des manchots. La cellule ci-dessous charge les données et les outils de tout le notebook : exécute-la d'abord.

In [ ]:
# Tools and data for the notebook exercises (parts A to D)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics as skm

penguins = wb.datasets.load_penguins()                    # the 344 penguins
MEASURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
measured = penguins.dropna(subset=MEASURES).reset_index(drop=True)   # the 342 penguins with their 4 measures
species = measured["species"].to_numpy()                  # their true species
flipper = measured["flipper_length_mm"].to_numpy()        # in mm (whole numbers)
is_gentoo = (species == "Gentoo").astype(int)             # 1 = Gentoo, 0 = another species (3.20, 3.24)


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")


def fr(value, decimals=2):
    """A number written the French way, for the messages: fr(0.25) -> '0,25'."""
    return f"{value:.{decimals}f}".replace(".", ",")


def run_metrics_tests(keyword, impl="learner"):
    """Run the tests of mylearn.metrics selected by `keyword` (on YOUR code by default)."""
    command = [sys.executable, "-m", "pytest", "tests/test_ch03_metrics.py", "-k", keyword, "-q",
               "-p", "no:cacheprovider", "--color=no", "-rf", "--tb=no"]
    if impl != "learner":
        command.append(f"--impl={impl}")
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                            env={**os.environ, "COLUMNS": "1000"})   # long lines: the reason of each failure
    lines = result.stdout.strip().splitlines()
    failed = [line for line in lines if line.startswith("FAILED ")]
    for line in failed[:8]:                                  # the test, then the reason of its failure
        name, _, reason = line.removeprefix("FAILED tests/test_ch03_metrics.py::").partition(" - ")
        print(f"❌ {name}\n   {reason[:400]}")
    if len(failed) > 8:
        print(f"   ... and {len(failed) - 8} other failed test(s)")
    print("pytest:", lines[-1] if lines else result.stderr.strip()[-300:])


def error_name(func, *args, **kwargs):
    """Name of the exception raised by func(*args, **kwargs), or "no error"."""
    try:
        func(*args, **kwargs)
    except NotImplementedError:
        raise
    except Exception as error:  # noqa: BLE001 - we want the name of whatever is raised
        return type(error).__name__
    return "no error"


print(f"{len(penguins)} penguins, {len(measured)} with their 4 measures")

### Ex 3.12 — Dix mille fléchettes : estimer des aires (et π) 🔬 ★ ⏱️ 15 min
**Objectif :** estimer une aire en comptant des fléchettes simulées, et mesurer comment l'erreur diminue avec le nombre de lancers.  
**Prérequis :** ch. 2 (loi uniforme, graine, écart-type, vitesse en $1/\sqrt{n}$) · fiche §3.2, §3.3 · livre §3.2, §3.3 · **Fil rouge :** synthétique · **Parcours :** M, C

Le mur est un carré de 2 m de côté, avec un disque de rayon 1 m peint en son centre. Une fléchette tombe en un point $(x, y)$ tiré uniformément sur le mur (fiche §3.2) ; elle touche le disque si $(x - 1)^2 + (y - 1)^2 \le 1$. D'après la fiche, la probabilité de toucher le disque est le rapport de son aire, $\pi$, à celle du mur, 4 : compter les fléchettes dans le disque **estime** donc $\pi$.

Pour que tout le monde obtienne les mêmes nombres, tire les points **exactement** ainsi : `points = 2 * rng.random((n, 2))` (une ligne par fléchette, colonnes $x$ et $y$), avec le générateur `rng` indiqué.

a) `pi_hat_12` : avec `rng = np.random.default_rng(3)` et $n = 10\,000$ fléchettes, l'estimation $\hat{\pi} = 4 \times$ (part des fléchettes tombées dans le disque) (3 décimales).

Écris ensuite `estimate_pi(n, rng)`, qui fait la même chose pour $n$ fléchettes et renvoie un `float`. La vérification contrôle qu'elle redonne la bonne réponse de a) avec la graine 3, et qu'avec un million de fléchettes elle tombe à moins de 0,01 de $\pi$.

b) `spread_12` : crée **un seul** générateur, `rng = np.random.default_rng(12)`, puis calcule 200 estimations de 10 000 fléchettes chacune, l'une après l'autre avec ce même générateur (`[estimate_pi(10_000, rng) for _ in range(200)]`). Donne leur écart-type (`np.std`, 3 décimales) : c'est l'erreur typique d'une estimation.
c) `factor_12` : par combien faut-il multiplier le nombre de fléchettes pour diviser cette erreur typique par 10 ? (un entier ; souviens-toi de la vitesse en $1/\sqrt{n}$ du ch. 2). Une fois c) rempli, la vérification le mesure sur d'autres nombres de fléchettes.

In [ ]:
pi_hat_12 = ...   # a)


def estimate_pi(n, rng):
    """Estimate pi with n darts thrown uniformly on the 2 m x 2 m wall: 4 x (share of darts in the disc)."""
    raise NotImplementedError("estimate_pi() is not written yet")


spread_12 = ...   # b)
factor_12 = ...   # c) an integer

In [ ]:
wb.check("3.12a", pi_hat_12)
wb.check("3.12b", spread_12)
wb.check("3.12c", factor_12)
with wb.attempt("3.12"):
    first_12 = estimate_pi(10_000, np.random.default_rng(3))
    verdict("3.12", isinstance(first_12, float) and bool(wb.check("3.12a", first_12, quiet=True)),
            "estimate_pi redonne la bonne réponse de a) avec la graine 3, sous forme de float.",
            "estimate_pi(10_000, np.random.default_rng(3)) doit redonner la bonne réponse de a), en float : tire les "
            "points exactement comme l'énoncé, et renvoie 4 fois la part des fléchettes dans le disque.")
    big_12 = estimate_pi(1_000_000, np.random.default_rng(0))
    verdict("3.12", abs(big_12 - np.pi) < 0.01, f"avec un million de fléchettes : {fr(big_12, 4)}, à moins de 0,01 de π.",
            f"avec un million de fléchettes, ton estimation vaut {fr(big_12, 4)} : trop loin de π.")
    if factor_12 is not ...:
        rng_demo = np.random.default_rng(0)
        for n_demo in [25, 250, 2_500]:
            spread_demo = np.std([estimate_pi(n_demo, rng_demo) for _ in range(500)])
            print(f"   {n_demo:>6} darts: typical error of an estimate ≈ {spread_demo:.4f}")

### Ex 3.13 — Deux disques : P(A|B) = P(B|A) ? 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir quand $P(A \mid B)$ et $P(B \mid A)$ sont égales, puis le vérifier avec des fléchettes.  
**Prérequis :** Ex 3.12, Ex 3.1 · fiche §3.4 · livre §3.4 · **Fil rouge :** synthétique · **Parcours :** M, C

Un mur carré de 4 m de côté porte deux disques qui se chevauchent en partie : $A$, de rayon 0,5 m, et $B$, de rayon 1 m. **Sans rien exécuter**, prévois :
a) `prediction_3_13a` : laquelle est la plus grande, $P(A \mid B)$ ou $P(B \mid A)$ ? Réponds `1` si c'est $P(A \mid B)$, `-1` si c'est $P(B \mid A)$, `0` si elles sont égales ;
b) `prediction_3_13b` : même question si l'on agrandit $A$ jusqu'au rayon de $B$, les deux disques se chevauchant toujours en partie ;
c) `prediction_3_13c` : dans la situation de départ, le rapport $\frac{P(A \mid B)}{P(B \mid A)}$ (2 décimales). Écris les deux probabilités comme des rapports d'aires (fiche §3.4) : l'aire de la partie commune, difficile à calculer, n'est pas nécessaire.

Écris ton hypothèse (cellule 📝), puis tes trois prédictions ; la vérification ne regarde que tes prédictions. Ensuite seulement, exécute l'**Expérience** : 100 000 fléchettes.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_3_13a = ...   # 1, -1 or 0
prediction_3_13b = ...   # 1, -1 or 0
prediction_3_13c = ...   # a number

In [ ]:
wb.check("3.13a", prediction_3_13a)
wb.check("3.13b", prediction_3_13b)
wb.check("3.13c", prediction_3_13c)

**Expérience** : exécute la cellule et compare avec tes prédictions.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_3_13a, prediction_3_13b, prediction_3_13c]):
    print("⏳ Ex 3.13 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    rng_13 = np.random.default_rng(13)
    darts_13 = 4 * rng_13.random((100_000, 2))                          # the 4 m x 4 m wall
    in_a = ((darts_13 - [1.5, 2.0]) ** 2).sum(axis=1) <= 0.5 ** 2      # disc A: centre (1.5, 2), radius 0.5
    in_b = ((darts_13 - [2.2, 2.0]) ** 2).sum(axis=1) <= 1.0 ** 2      # disc B: centre (2.2, 2), radius 1
    p_a_given_b = np.sum(in_a & in_b) / np.sum(in_b)
    p_b_given_a = np.sum(in_a & in_b) / np.sum(in_a)
    print(f"darts in A: {in_a.sum()} · in B: {in_b.sum()} · in A and B: {np.sum(in_a & in_b)}")
    print(f"P(A | B) ≈ {p_a_given_b:.3f}   P(B | A) ≈ {p_b_given_a:.3f}")

    shown = slice(0, 4000)                                              # 4 000 darts are enough for the picture
    colour = np.where(in_a & in_b, "C2", np.where(in_a, "C0", np.where(in_b, "C1", "0.8")))[shown]
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.scatter(*darts_13[shown].T, s=3, c=colour)
    ax.add_patch(plt.Circle((1.5, 2.0), 0.5, fill=False, color="C0", lw=2))
    ax.add_patch(plt.Circle((2.2, 2.0), 1.0, fill=False, color="C1", lw=2))
    ax.text(1.15, 2.6, "A", color="C0", fontsize=14)
    ax.text(2.9, 2.9, "B", color="C1", fontsize=14)
    ax.set_xlim(0, 4)
    ax.set_ylim(0, 4)
    ax.set_aspect("equal")
    ax.set_title("4 000 of the 100 000 darts")
    plt.show()

d) `ratio_13` : le rapport mesuré $\frac{P(A \mid B)}{P(B \mid A)}$, calculé avec les variables de l'expérience (2 décimales). Compare-le à ta prédiction c. Dans tes notes : ce rapport d'estimations est exactement un rapport de deux comptages de fléchettes ; lesquels ?

In [ ]:
ratio_13 = ...   # d)

In [ ]:
wb.check("3.13d", ratio_13)

### Ex 3.14 — Penguins : espèce × île avec pd.crosstab 📦 ★★ ⏱️ 20 min
**Objectif :** lire des probabilités jointes, marginales et conditionnelles dans une table de contingence, et tester l'indépendance de deux variables.  
**Prérequis :** Ex 3.3, rappel 3.R3 · fiche §3.4 à §3.6 (🧮 Rappel outil : `pd.crosstab`) · **Fil rouge :** Penguins · **Parcours :** C

La fiche a croisé le sexe et l'espèce des manchots (§3.6). Fais de même avec l'**espèce** et l'**île**, pour les 344 manchots de `penguins` (aucune de ces deux colonnes ne manque).

a) `table_14` : la table de contingence (`pd.crosstab`), espèces en lignes, îles en colonnes, **sans** les marges ;
b) `p_gentoo_given_biscoe` : $P(\text{Gentoo} \mid \text{Biscoe})$ (3 décimales) ;
c) `p_dream_given_chinstrap` : $P(\text{Dream} \mid \text{Chinstrap})$ (2 décimales) ;
d) `p_adelie_and_dream` : la probabilité jointe $P(\text{Adelie}, \text{Dream})$ (3 décimales) ;
e) `p_adelie_given_dream` : $P(\text{Adelie} \mid \text{Dream})$ (3 décimales) ;
f) `independent_14` : l'espèce et l'île sont-elles indépendantes ? (`True` ou `False` ; justifie-le dans tes notes par un calcul, fiche §3.6) ;
g) `normalize_14` : la valeur de l'argument `normalize` de `pd.crosstab` qui donne, dans chaque **ligne** de la table, $P(\text{île} \mid \text{espèce})$ (une chaîne).

Calcule b à e à partir de la table (`.loc`, sommes de lignes ou de colonnes), pas en recopiant des nombres à la main : c'est ainsi qu'on évite les fautes de frappe.

In [ ]:
table_14 = ...                  # a) species in rows, islands in columns
p_gentoo_given_biscoe = ...     # b)
p_dream_given_chinstrap = ...   # c)
p_adelie_and_dream = ...        # d)
p_adelie_given_dream = ...      # e)
independent_14 = ...            # f) True or False
normalize_14 = ...              # g) a string

In [ ]:
for letter, answer in zip("abcdefg", [table_14, p_gentoo_given_biscoe, p_dream_given_chinstrap, p_adelie_and_dream,
                                     p_adelie_given_dream, independent_14, normalize_14]):
    wb.check(f"3.14{letter}", answer)

## Partie B · La matrice de confusion et les mesures binaires

Fiche §3.7.1 à §3.7.11. Tu écris le cœur de `mylearn.metrics` : la matrice de confusion (3.15), puis accuracy, precision, recall, F-beta et F1 dans le cas binaire (3.16), et enfin le tableau complet des mesures (3.19). Entre les deux : une matrice lue à l'envers (3.17) et deux classifieurs extrêmes (3.18). La cellule ci-dessous prépare les données de la partie : les prédictions des règles de la biologiste (1.15) pour les 342 manchots, et les réponses d'un filtre anti-spam testé sur 300 e-mails.

In [ ]:
def expert_rule(bill_length, bill_depth, flipper_length, body_mass):
    """The biologist's three rules of exercise 1.15: returns "Gentoo", "Chinstrap" or "Adelie"."""
    if body_mass > 4700:
        return "Gentoo"
    if bill_length > 45:
        return "Chinstrap"
    return "Adelie"


def predict_with(rule, X):
    """Apply rule(bill_length, bill_depth, flipper_length, body_mass) to every row of the DataFrame X."""
    return np.array([rule(*row) for row in X[MEASURES].to_numpy()])


expert_pred = predict_with(expert_rule, measured)       # the species given by the rules, for the 342 penguins

# 3.16 and 3.19: a spam filter tested on 300 e-mails (labels "spam" and "ham")
rng_spam = np.random.default_rng(16)
is_spam = rng_spam.random(300) < 0.25
u_spam = rng_spam.random(300)
spam_true = np.where(is_spam, "spam", "ham")                                          # the truth
spam_pred = np.where(np.where(is_spam, u_spam < 0.70, u_spam < 0.06), "spam", "ham")  # the filter's answers
print(f"{len(expert_pred)} predictions of the expert rules · {len(spam_true)} e-mails for the spam filter")

### Ex 3.15 — confusion_matrix à la manière de scikit-learn 🔨 ★★ ⏱️ 20 min
**Objectif :** écrire la matrice de confusion, vérité en lignes et prédictions en colonnes, comme scikit-learn.  
**Prérequis :** Ex 3.2 · Ex 1.15 (les règles de la biologiste) · fiche §3.7.1, §3.7.2 (🕰️ conventions) · **Fil rouge :** Penguins · **mylearn :** `metrics.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/metrics.py` (créé par `python tools/start_chapter.py 3`), lis la docstring de chaque fonction, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy est permis (`np.asarray`, `np.unique`, `np.concatenate`, `np.sum`, `np.argsort`, `np.cumsum`…), scikit-learn non : `sklearn.metrics` est l'**oracle** des tests. Écris une petite fonction d'aide, par exemple `_check_pair(y_true, y_pred)`, qui convertit les deux entrées (`np.asarray`), lève une `ValueError` si leurs longueurs diffèrent ou si elles sont vides, et les renvoie : toutes les fonctions de mesure l'appelleront. Une mesure se renvoie en `float` Python (`float(...)` ; les tests acceptent aussi un `np.float64`). La cellule de vérification recharge ta librairie, vérifie quelques valeurs, puis lance les tests de tes fonctions ; `python -m pytest tests/test_ch03_metrics.py -q` les lance tous dans un terminal.

Écris `confusion_matrix(y_true, y_pred, labels=None)` (lis sa docstring) :
- sans `labels`, les étiquettes sont celles qu'on trouve dans `y_true` **et** dans `y_pred`, triées : `np.unique` sur les deux réunies (`np.concatenate`) ;
- la case `C[i, j]` compte les échantillons de vérité `labels[i]` prédits `labels[j]` : un dictionnaire `{étiquette: numéro}` donne la ligne et la colonne de chaque échantillon, puis une boucle (ou `np.add.at`) compte ;
- lève une `ValueError` si les longueurs diffèrent, si les entrées sont vides, ou si une étiquette des données manque dans `labels` (scikit-learn, lui, écarterait ces échantillons sans rien dire).

Vérifications, sur les 342 manchots (la cellule de vérification appelle ta fonction : il n'y a rien à recopier) :
a) la matrice de confusion des règles de la biologiste (`species` contre `expert_pred`) ;
b) la même, avec les lignes et les colonnes dans l'ordre `["Gentoo", "Chinstrap", "Adelie"]` ;
c) `gentoo_as_chinstrap` : combien de Gentoo les règles prennent-elles pour des Chinstrap ? Lis-le dans la matrice que la vérification affiche (un entier).

Puis les tests de `confusion_matrix`.

In [ ]:
gentoo_as_chinstrap = ...   # c) read it in the matrix displayed by the check (an integer)

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="3")   # reload your saved file
with wb.attempt("3.15"):
    C_15 = mylearn.metrics.confusion_matrix(species, expert_pred)
    wb.check("3.15a", C_15, computed=True)
    wb.check("3.15b", mylearn.metrics.confusion_matrix(species, expert_pred, labels=["Gentoo", "Chinstrap", "Adelie"]), computed=True)
    verdict("3.15", error_name(mylearn.metrics.confusion_matrix, species, expert_pred, labels=["Adelie", "Gentoo"]) == "ValueError",
            "une étiquette absente de labels lève bien une ValueError.",
            "confusion_matrix(species, expert_pred, labels=[\"Adelie\", \"Gentoo\"]) doit lever une ValueError : les "
            "Chinstrap n'ont ni ligne ni colonne.")
    wb.plot.plot_confusion_matrix(C_15, class_names=["Adelie", "Chinstrap", "Gentoo"], title="Expert rules (1.15), 342 penguins")
    plt.show()
    run_metrics_tests("test_confusion_matrix_")
wb.check("3.15c", gentoo_as_chinstrap)

### Ex 3.16 — accuracy, precision, recall, F-beta et F1 (cas binaire) 🔨 ★★★ ⏱️ 40 min
**Objectif :** écrire accuracy, precision, recall, F-beta et F1 pour deux classes, avec une classe positive au choix.  
**Prérequis :** Ex 3.15 · fiche §3.7.5 à §3.7.11 · **Fil rouge :** synthétique · **mylearn :** `metrics.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/metrics.py`, mêmes règles qu'en 3.15 (NumPy permis, scikit-learn non). Enregistre, puis relance la cellule de vérification.

Écris `accuracy`, `precision`, `recall`, `fbeta` et `f1` pour le cas **binaire** (`average="binary"`, la valeur par défaut) ; les autres valeurs d'`average` viendront en 3.25 : d'ici là, fais-leur lever une `NotImplementedError`. Relis les docstrings :
- la classe positive est `pos_label`. En binaire, il y a au plus deux étiquettes, cherchées dans `y_true` **et** dans `y_pred` ; s'il y en a deux, `pos_label` doit en faire partie (sinon `ValueError`) ;
- `zero_division` est la valeur renvoyée quand un dénominateur est nul : aucune prédiction positive (precision), aucun positif réel (recall), $TP + FN + FP = 0$ (F-beta) ;
- $F_\beta = \frac{(1 + \beta^2)\,TP}{(1 + \beta^2)\,TP + \beta^2\,FN + FP}$ (fiche §3.7.11) ; `ValueError` si $\beta \le 0$, et `f1` appelle `fbeta` avec $\beta = 1$.

Compte TP, FP et FN dans **une** fonction d'aide que les quatre mesures partagent : moins de code, moins de bugs.

Vérifications, sur le filtre anti-spam (`spam_true`, `spam_pred` ; étiquettes `"spam"` et `"ham"`, classe positive `"spam"`) ; la cellule de vérification appelle tes fonctions :
a) l'accuracy ; b) la precision ; c) le recall ; d) le F1 ; e) le $F_2$ (`fbeta` avec `beta=2`) ;
puis un contrôle : sans `pos_label`, `precision(spam_true, spam_pred)` doit lever une `ValueError`, car l'étiquette positive par défaut, 1, n'existe pas ici ; enfin, les tests des cinq fonctions (cas binaire).

Dans tes notes : le $F_2$ est-il plus proche de la precision ou du recall ? Pourquoi ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="3")   # reload your saved file
with wb.attempt("3.16"):
    mm = mylearn.metrics
    wb.check("3.16a", mm.accuracy(spam_true, spam_pred), computed=True)
    wb.check("3.16b", mm.precision(spam_true, spam_pred, pos_label="spam"), computed=True)
    wb.check("3.16c", mm.recall(spam_true, spam_pred, pos_label="spam"), computed=True)
    wb.check("3.16d", mm.f1(spam_true, spam_pred, pos_label="spam"), computed=True)
    wb.check("3.16e", mm.fbeta(spam_true, spam_pred, beta=2, pos_label="spam"), computed=True)
    verdict("3.16", error_name(mm.precision, spam_true, spam_pred) == "ValueError",
            "sans pos_label, la classe positive 1 n'existe pas : ValueError, comme scikit-learn.",
            "precision(spam_true, spam_pred) doit lever une ValueError : pos_label=1 n'est pas une des étiquettes.")
    run_metrics_tests("(test_accuracy_ or test_precision_ or test_recall_ or test_fbeta_ or test_f1_) and not multiclass and not curve")

### Ex 3.17 — La matrice à l'envers 🐛 ★★ ⏱️ 15 min
**Objectif :** diagnostiquer une matrice de confusion lue dans le mauvais sens, puis écrire une version juste quelles que soient les étiquettes.  
**Prérequis :** Ex 3.16 · fiche §3.7.2 (🕰️ conventions de scikit-learn), §3.7.3 · **Fil rouge :** synthétique · **Parcours :** C

Un laboratoire évalue un test de dépistage rapide sur 2 000 personnes : `test_truth` contient la vérité (`"malade"` ou `"sain"`) et `test_result` le résultat du test. Un collègue a écrit `screening_report`, qui renvoie la sensibilité (le recall des malades) et la precision (la part de malades parmi les tests positifs). Son code tourne, ses chiffres semblent plausibles… et ils sont faux.

a) `true_sensitivity_17` : la vraie sensibilité du test, calculée avec **tes** fonctions de `mylearn.metrics` (3 décimales) ;
b) `true_precision_17` : sa vraie precision, avec tes fonctions aussi (3 décimales) ;
c) `top_left_17` : dans `skm.confusion_matrix(test_truth, test_result)`, quelle case se trouve en haut à gauche : `"TP"`, `"FN"`, `"FP"` ou `"TN"` (la classe positive étant `"malade"`) ? C'est la clé du bug.

Puis corrige : écris `screening_report_fixed(y_true, y_pred, positive)`, qui renvoie `{"sensitivity": ..., "precision": ...}`, juste quelles que soient les étiquettes, la classe positive étant `positive`. La vérification l'essaie avec quatre jeux d'étiquettes. Dans tes notes : quelles mesures le collègue a-t-il calculées en réalité ? Pourquoi son code marche-t-il avec des étiquettes 0 et 1, et pas ici ? Comment l'aurais-tu repéré ?

In [ ]:
rng_17 = np.random.default_rng(17)
sick_17 = rng_17.random(2000) < 0.10
u_17 = rng_17.random(2000)
test_truth = np.where(sick_17, "malade", "sain")                                          # the truth
test_result = np.where(np.where(sick_17, u_17 < 0.85, u_17 < 0.05), "malade", "sain")   # the rapid test


def screening_report(y_true, y_pred):
    """Sensitivity and precision of a screening test (the colleague's version)."""
    tn, fp, fn, tp = skm.confusion_matrix(y_true, y_pred).ravel()
    return {"sensitivity": tp / (tp + fn), "precision": tp / (tp + fp)}


print("the colleague's report:", {key: round(float(value), 3) for key, value in screening_report(test_truth, test_result).items()})

In [ ]:
true_sensitivity_17 = ...   # a) with your mylearn.metrics functions
true_precision_17 = ...     # b)
top_left_17 = ...           # c) "TP", "FN", "FP" or "TN"


def screening_report_fixed(y_true, y_pred, positive):
    """{"sensitivity": ..., "precision": ...} for any labels; `positive` is the positive class."""
    raise NotImplementedError("screening_report_fixed() is not written yet")

In [ ]:
wb.check("3.17a", true_sensitivity_17)
wb.check("3.17b", true_precision_17)
wb.check("3.17c", top_left_17)
with wb.attempt("3.17"):
    sick_truth, sick_result = test_truth == "malade", test_result == "malade"
    cases_17 = {"malade / sain": (test_truth, test_result, "malade"),
                "1 / 0": (sick_truth.astype(int), sick_result.astype(int), 1),
                "yes / no": (np.where(sick_truth, "yes", "no"), np.where(sick_result, "yes", "no"), "yes"),
                "fraude / normal": (np.where(sick_truth, "fraude", "normal"), np.where(sick_result, "fraude", "normal"), "fraude")}
    for name, (truth, result, positive) in cases_17.items():
        report_17 = screening_report_fixed(truth, result, positive)
        expected_17 = (skm.recall_score(truth, result, pos_label=positive), skm.precision_score(truth, result, pos_label=positive))
        verdict("3.17", np.allclose([report_17["sensitivity"], report_17["precision"]], expected_17),
                f"étiquettes {name} : sensibilité et precision justes.",
                f"étiquettes {name} : ta sensibilité ou ta precision est fausse.")

### Ex 3.18 — Tout positif, un seul positif : prédire les scores 🔮 ★★ ⏱️ 15 min
**Objectif :** prévoir l'accuracy, le F1 et la balanced accuracy de deux classifieurs extrêmes, et voir que ces mesures ne les classent pas dans le même ordre.  
**Prérequis :** Ex 3.16 · fiche §3.7.9 (balanced accuracy), §3.7.10 · **Fil rouge :** synthétique · **Parcours :** C

Un jeu de test de 40 échantillons contient 8 positifs. Deux classifieurs paresseux :
- le classifieur 1 répond « positif » pour **tous** les échantillons ;
- le classifieur 2 ne répond « positif » que pour **un seul** échantillon, le plus évident, qui est bien un positif ; « négatif » pour tous les autres.

La fiche (§3.7.10) donne déjà leur precision et leur recall. **Sans rien exécuter**, prévois :
a) `prediction_3_18a` : l'accuracy du classifieur 1 (2 décimales) ;
b) `prediction_3_18b` : son F1 (3 décimales) ;
c) `prediction_3_18c` : l'accuracy du classifieur 2 (3 décimales) ;
d) `prediction_3_18d` : son F1 (3 décimales) ;
e) `prediction_3_18e` : lequel a la meilleure balanced accuracy (fiche §3.7.9) : `1` ou `2` ?

Écris ton hypothèse (cellule 📝), puis tes cinq prédictions ; la vérification ne regarde que tes prédictions. Ensuite seulement, exécute l'**Expérience**.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_3_18a = ...   # the accuracy of classifier 1
prediction_3_18b = ...   # its F1
prediction_3_18c = ...   # the accuracy of classifier 2
prediction_3_18d = ...   # its F1
prediction_3_18e = ...   # 1 or 2

In [ ]:
for letter, prediction in zip("abcde", [prediction_3_18a, prediction_3_18b, prediction_3_18c,
                                        prediction_3_18d, prediction_3_18e]):
    wb.check(f"3.18{letter}", prediction)

**Expérience** : exécute la cellule et compare avec tes prédictions.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_3_18a, prediction_3_18b, prediction_3_18c, prediction_3_18d, prediction_3_18e]):
    print("⏳ Ex 3.18 : écris d'abord tes cinq prédictions, puis relance cette cellule.")
else:
    y_18 = np.array([1] * 8 + [0] * 32)          # 8 positives, then 32 negatives
    all_positive = np.ones(40, dtype=int)          # classifier 1
    one_positive = np.zeros(40, dtype=int)         # classifier 2: only the most obvious case, a real positive
    one_positive[0] = 1
    for name, pred in [("1: all positive", all_positive), ("2: one positive", one_positive)]:
        print(f"{name:<16} accuracy {skm.accuracy_score(y_18, pred):.3f} · F1 {skm.f1_score(y_18, pred):.3f} · "
              f"balanced accuracy {skm.balanced_accuracy_score(y_18, pred):.4f} · "
              f"precision {skm.precision_score(y_18, pred):.3f} · recall {skm.recall_score(y_18, pred):.3f}")

### Ex 3.19 — Le tableau de bord complet : classification_rates 🔨 ★★ ⏱️ 20 min
**Objectif :** calculer d'un coup les 13 mesures d'une matrice binaire, avec une seule règle pour les divisions par zéro.  
**Prérequis :** Ex 3.16, Ex 3.5 · fiche §3.7.9 (tableau et 🕰️) · **Fil rouge :** synthétique · **mylearn :** `metrics.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/metrics.py`, mêmes règles qu'en 3.15 (NumPy permis, scikit-learn non). Enregistre, puis relance la cellule de vérification.

Écris `classification_rates(y_true, y_pred, pos_label=1, zero_division=0.0)`, qui renvoie un dictionnaire des 13 mesures du tableau de la fiche (§3.7.9), **dans l'ordre de la docstring** (un dictionnaire Python garde l'ordre d'insertion). Compte TP, FN, FP et TN une seule fois, puis écris une petite fonction d'aide `ratio(num, den)` qui renvoie `zero_division` quand `den` vaut 0 : **tous** les rapports passent par elle, ceux du MCC et de la balanced accuracy compris (scikit-learn a pour ces deux-là ses propres conventions ; les tests suivent la docstring). Mêmes contrôles des étiquettes qu'en 3.16 ; chaque valeur est un `float`.

Vérifications, sur le filtre anti-spam de 3.16 (classe positive `"spam"`) :
a) `[specificity, npv, balanced_accuracy, mcc]`, calculés par ta fonction (la cellule de vérification l'appelle) ;
b) `swap_19` : si l'on prend `"ham"` comme classe positive, la precision devient l'une des autres mesures du tableau calculé avec `"spam"` : laquelle ? (sa clé, une chaîne) ;
puis l'ordre des clés, et les tests de `classification_rates`.

In [ ]:
swap_19 = ...   # b) a key of the dictionary

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="3")   # reload your saved file
with wb.attempt("3.19"):
    rates_19 = mylearn.metrics.classification_rates(spam_true, spam_pred, pos_label="spam")
    if {"specificity", "npv", "balanced_accuracy", "mcc"} <= set(rates_19):
        wb.check("3.19a", [rates_19["specificity"], rates_19["npv"], rates_19["balanced_accuracy"], rates_19["mcc"]],
                 computed=True)
    verdict("3.19", list(rates_19) == ["accuracy", "balanced_accuracy", "precision", "recall", "specificity", "npv", "fpr", "fnr", "fdr", "false_omission_rate", "prevalence", "f1", "mcc"],
            "les 13 clés sont là, dans l'ordre de la docstring.",
            "les clés doivent être exactement celles de la docstring, dans le même ordre.")
    run_metrics_tests("test_classification_rates_")
wb.check("3.19b", swap_19)

## Partie C · Seuil, prévalence et outils de scikit-learn

Fiche §3.7.4, §3.7.10 et §3.8. Un classifieur donne souvent un **score** ; le seuil qui en fait une décision se choisit selon le coût des erreurs (3.20). La precision dépend de la prévalence : tu le vérifies par la simulation (3.21). Enfin, tu contrôles tes résultats avec les outils de scikit-learn et tu apprends à lire leur documentation (3.22, 3.23).

### Ex 3.20 — Un seuil sur la nageoire : precision et recall en balance 🔬 ★★ ⏱️ 25 min
**Objectif :** balayer les seuils d'un score, puis choisir un seuil selon le F1 ou selon le coût des erreurs.  
**Prérequis :** Ex 3.16 · fiche §3.7.4, §3.7.10 (🧮 score et seuil), §3.7.11 · **Fil rouge :** Penguins · **Parcours :** R, M, C

Pas encore de modèle entraîné (ce sera au ch. 7) : le **score** est ici une simple mesure, la longueur de la nageoire (`flipper`, en mm, des nombres entiers). La classe positive est « Gentoo » (`is_gentoo` : 1 pour un Gentoo, 0 sinon), contre les deux autres espèces. Au seuil $t$, on prédit « Gentoo » quand `flipper >= t` (supérieur **ou égal**).

Écris d'abord `sweep_20(thresholds)`, qui renvoie un DataFrame avec une ligne par seuil et les colonnes `threshold`, `precision`, `recall`, `f1`, `fn` et `fp`, calculées avec tes fonctions de `mylearn.metrics` (les prédictions en 0/1 : `(flipper >= t).astype(int)`). La vérification trace precision, recall et F1 en fonction du seuil, avec ton tableau `sweep_20(np.arange(190, 231))`. Puis :

a) `pr_205` : `[precision, recall]` au seuil $t = 205$ mm (3 décimales) ;
b) `best_f1_t` : parmi les seuils entiers de 190 à 230 mm, celui qui maximise le F1 ;
c) `best_cost_t` : un Gentoo manqué (FN) coûte 10, un faux Gentoo (FP) coûte 1 ; parmi les mêmes seuils, celui qui minimise le coût total $10 \times FN + 1 \times FP$.

Lis b et c dans ton tableau (`idxmax`, `idxmin`, 0A). Dans tes notes : pourquoi les seuils de b et c diffèrent-ils ? Lequel choisirais-tu pour un recensement où rater un Gentoo coûte cher ?

In [ ]:
def sweep_20(thresholds):
    """One row per threshold t (Gentoo when flipper >= t): threshold, precision, recall, f1, fn, fp."""
    raise NotImplementedError("sweep_20() is not written yet")


pr_205 = ...        # a) [precision, recall] at t = 205
best_f1_t = ...     # b)
best_cost_t = ...   # c)

In [ ]:
wb.check("3.20a", pr_205)
wb.check("3.20b", best_f1_t)
wb.check("3.20c", best_cost_t)
with wb.attempt("3.20"):
    table_20 = sweep_20(np.arange(190, 231))
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    for column in ["precision", "recall", "f1"]:
        ax.plot(table_20["threshold"], table_20[column], marker=".", label=column)
    ax.set_xlabel("threshold t (mm): Gentoo when flipper >= t")
    ax.set_ylim(0, 1.02)
    ax.legend()
    plt.show()

### Ex 3.21 — Simuler le dépistage : la prévalence fait la precision 🔬 ★★ ⏱️ 30 min
**Objectif :** vérifier par la simulation que la precision d'un test dépend de la prévalence, et trouver la prévalence à partir de laquelle un positif est plus souvent malade que sain.  
**Prérequis :** Ex 3.7, Ex 3.19 · ch. 2 (tirages de Bernoulli, graine) · fiche §3.8 · **Fil rouge :** synthétique · **Parcours :** M, C

Le test du livre (§3.8) a une sensibilité de 0,99 et une spécificité de 0,98 ; 1 % de la population est malade. Tu as calculé ses mesures sur papier (3.7, 2ᵉ partie) ; ici, tu mesures sa precision sur une population simulée.

Écris `simulate_screening(n, prevalence, sensitivity, specificity, rng)`, qui renvoie deux tableaux de booléens `(sick, positive)` de longueur $n$, tirés **exactement** ainsi (pour que tout le monde obtienne les mêmes nombres) :
1. `sick = rng.random(n) < prevalence` ;
2. puis `u = rng.random(n)` : une personne est positive si `u < sensitivity` quand elle est malade, si `u < 1 - specificity` quand elle est saine (`np.where`).

a) `precision_21` : la precision mesurée sur $n = 100\,000$ personnes, avec `rng = np.random.default_rng(21)` (4 décimales) ; par exemple avec ta fonction `classification_rates` de 3.19 (convertis d'abord les booléens en 0/1 avec `.astype(int)`), ou directement en comptant ;
b) écris `theoretical_precision(prevalence, sensitivity, specificity)`, la precision attendue (fréquences naturelles, fiche §3.8), puis `theory_21` : ses valeurs pour une prévalence de 0,1 % et de 10 % (liste, 3 décimales) ;
c) `half_21` : la prévalence pour laquelle la precision vaut exactement 0,5 (4 décimales) : un positif a alors autant de chances d'être malade que sain. Résous l'équation à la main, ou cherche-la sur une grille de prévalences (un pas d'au plus 0,000 01, et la prévalence dont la precision est la plus proche de 0,5) ;
d) `double_21` : juste après le premier test (avec le même générateur, sans le recréer), fais passer un **second** test indépendant, de mêmes caractéristiques, à toute la population : `u2 = rng.random(n)`, puis la même règle qu'au point 2. Quelle est la precision parmi les personnes positives **aux deux** tests (3 décimales) ?

La vérification trace la precision théorique en fonction de la prévalence, avec ton point simulé. Dans tes notes : pourquoi un second test change-t-il autant la precision ? (Le ch. 4, règle de Bayes, y reviendra.)

In [ ]:
def simulate_screening(n, prevalence, sensitivity, specificity, rng):
    """(sick, positive): two boolean arrays of length n, drawn exactly as in the statement."""
    raise NotImplementedError("simulate_screening() is not written yet")


def theoretical_precision(prevalence, sensitivity, specificity):
    """Expected precision P(sick | positive) of the test (natural frequencies)."""
    raise NotImplementedError("theoretical_precision() is not written yet")


precision_21 = ...   # a)
theory_21 = ...      # b) [prevalence 0.1 %, prevalence 10 %]
half_21 = ...        # c)
double_21 = ...      # d)

In [ ]:
wb.check("3.21a", precision_21)
wb.check("3.21b", theory_21)
wb.check("3.21c", half_21)
wb.check("3.21d", double_21)
with wb.attempt("3.21"):
    sick_21, positive_21 = simulate_screening(100_000, 0.01, 0.99, 0.98, np.random.default_rng(21))
    verdict("3.21", bool(wb.check("3.21a", np.sum(sick_21 & positive_21) / np.sum(positive_21), quiet=True)),
            "simulate_screening tire comme l'énoncé : sa precision est la bonne réponse de a).",
            "avec la graine 21, simulate_screening ne redonne pas la bonne precision : tire exactement comme l'énoncé "
            "(les malades d'abord, puis un seul tableau u pour le test).")
    verdict("3.21", bool(wb.check("3.21b", [theoretical_precision(0.001, 0.99, 0.98), theoretical_precision(0.1, 0.99, 0.98)], quiet=True)),
            "theoretical_precision donne les bonnes valeurs de b).",
            "theoretical_precision ne donne pas les bonnes valeurs de b) : suis l'arbre des fréquences naturelles.")
    prevalences_21 = np.logspace(-4, np.log10(0.5), 200)
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    ax.plot(prevalences_21, [theoretical_precision(p, 0.99, 0.98) for p in prevalences_21], label="theory")
    ax.scatter([0.01], [np.sum(sick_21 & positive_21) / np.sum(positive_21)], color="C1", zorder=3,
               label="simulation, 100 000 people")
    ax.set_xscale("log")
    ax.set_xlabel("prevalence (share of sick people), log scale")
    ax.set_ylabel("precision P(sick | positive)")
    ax.set_title("The book's test: sensitivity 0.99, specificity 0.98")
    ax.legend()
    plt.show()

### Ex 3.22 — Vérifier avec scikit-learn : classification_report et affichages 📦 ★★ ⏱️ 20 min
**Objectif :** lire un rapport de classification et une matrice de confusion normalisée de scikit-learn.  
**Prérequis :** Ex 3.16, Ex 3.15 · fiche « Au-delà du livre : plusieurs classes », §3.7.9 (🕰️) · **Fil rouge :** Penguins · **Parcours :** R, C

scikit-learn résume tout en un tableau : `classification_report`. Applique-le aux règles de la biologiste (`species` contre `expert_pred`), avec `digits=3`, et affiche aussi leur matrice de confusion avec `skm.ConfusionMatrixDisplay.from_predictions(...)`, suivi de `plt.show()`.

a) `recall_gentoo_22` : le recall des Gentoo, lu dans le rapport (3 décimales) ;
b) `macro_f1_22` : le F1 macro (3 décimales) ;
c) `weighted_f1_22` : le F1 pondéré (3 décimales) ;
d) `normalize_22` : la valeur de l'argument `normalize` de `ConfusionMatrixDisplay.from_predictions` qui fait apparaître sur la diagonale le recall de chaque espèce (une chaîne ; lis sa documentation : `help(skm.ConfusionMatrixDisplay.from_predictions)`).

Récupère enfin le rapport sous forme de dictionnaire, `output_dict=True`, dans `report_22` : la vérification y lit le recall des Gentoo. Dans tes notes : pourquoi le F1 macro est-il plus bas que le F1 pondéré ? Quelle espèce le tire vers le bas ?

In [ ]:
# Display the report and the confusion matrix here, then fill in:
recall_gentoo_22 = ...   # a)
macro_f1_22 = ...        # b)
weighted_f1_22 = ...     # c)
normalize_22 = ...       # d) a string
report_22 = ...          # the report as a dictionary (output_dict=True)

In [ ]:
wb.check("3.22a", recall_gentoo_22)
wb.check("3.22b", macro_f1_22)
wb.check("3.22c", weighted_f1_22)
wb.check("3.22d", normalize_22)
if report_22 is ...:
    print("⏳ Ex 3.22 : report_22 pas encore fait.")
else:
    gentoo_22 = report_22.get("Gentoo", {}) if isinstance(report_22, dict) else {}
    verdict("3.22", bool(wb.check("3.22a", gentoo_22.get("recall"), quiet=True)),
            "report_22[\"Gentoo\"][\"recall\"] est bien le recall des Gentoo.",
            "report_22 doit être le dictionnaire de classification_report(..., output_dict=True), avec une entrée par espèce.")

### Ex 3.23 — Lire la documentation de sklearn.metrics 🛠️ ★★ ⏱️ 25 min
**Objectif :** trouver dans la documentation officielle le comportement exact d'une fonction avant de s'y fier.  
**Prérequis :** Ex 3.22 · fiche (🕰️, « Pour aller plus loin ») · **Parcours :** C

Réflexe professionnel : avant de faire confiance à un chiffre, vérifie **ce que calcule exactement** la fonction. La documentation de chaque fonction de `sklearn.metrics` décrit ses paramètres ; lis-la pour la version du workbook (1.6) : sur le site de scikit-learn, choisis la version en haut de la page, ou, dans le notebook, `help(skm.precision_score)`.

Pour chaque question, **prévois** la réponse en lisant la documentation, sans exécuter l'appel ; écris-la dans la variable, puis exécute la vérification, qui compare ta prévision au comportement réel de scikit-learn.
a) `zero_division` : que renvoie `skm.precision_score([1, 1, 0], [0, 0, 0], zero_division=1)` ? (un nombre)
b) `labels` : que renvoie `skm.confusion_matrix(["a", "b", "c", "a"], ["a", "c", "b", "a"], labels=["a", "b"])` ? (une liste de listes) Que deviennent les échantillons dont une étiquette n'est pas dans `labels` ? Ta `confusion_matrix` fait autrement : comment, et pourquoi ?
c) `pos_label` : quelle erreur lève `skm.recall_score(["spam", "ham", "spam"], ["spam", "spam", "ham"])` ? (son nom, une chaîne comme `"KeyError"`)
d) `normalize` : avec `skm.confusion_matrix(y_true, y_pred, normalize="pred")`, qu'est-ce qui somme à 1 : les `"lignes"`, les `"colonnes"` ou `"tout"` ?
e) `sample_weight` : que renvoie `skm.accuracy_score([1, 0, 1], [1, 1, 1], sample_weight=[1, 1, 2])` ? (un nombre)
f) `average=None` : que renvoie `skm.precision_score(["b", "a", "c", "a"], ["b", "a", "a", "c"], average=None)` ? (une liste de nombres) Dans quel ordre sont rangées les classes ?

Dans ta cellule 📝, recopie pour chaque question la phrase de la documentation qui y répond.

In [ ]:
answer_23a = ...   # a) a number
answer_23b = ...   # b) a list of lists
answer_23c = ...   # c) the name of the error, a string
answer_23d = ...   # d) "lignes", "colonnes" or "tout"
answer_23e = ...   # e) a number
answer_23f = ...   # f) a list of numbers

In [ ]:
def numbers_23(value):
    """The numbers of an answer written as a learner would: 0.75, "0,75", "3/4", or lists of them."""
    if isinstance(value, str):
        text = value.strip().replace(",", ".")
        if "/" in text:
            top, bottom = text.split("/")
            return float(top) / float(bottom)
        return float(text)
    if isinstance(value, (list, tuple, np.ndarray, pd.Series)):
        return [numbers_23(item) for item in value]
    return float(value)


WORDS_23 = {"column": "colonnes", "columns": "colonnes", "colonne": "colonnes", "colonnes": "colonnes",
            "row": "lignes", "rows": "lignes", "ligne": "lignes", "lignes": "lignes",
            "all": "tout", "tout": "tout", "toute": "tout", "total": "tout"}


def compare_23(letter, answer, actual, explanation):
    """Compare your prediction with what scikit-learn really does."""
    if answer is ... or answer is None:
        print(f"⏳ Ex 3.23{letter} : pas encore fait.")
        return
    try:
        mine, real = np.asarray(numbers_23(answer), dtype=float), np.asarray(numbers_23(actual), dtype=float)
        ok = mine.shape == real.shape and bool(np.allclose(mine, real))
    except (TypeError, ValueError, ZeroDivisionError):
        def word(text):
            text = str(text).strip().strip("'\"").lower()
            return WORDS_23.get(text, text)
        ok = word(answer) == word(actual)
    verdict(f"3.23{letter}", ok, f"juste ({explanation}).",
            f"scikit-learn donne {actual!r} ({explanation}) : relis la documentation.")


def raised_23(func, *args, **kwargs):
    """Name of the error raised by a scikit-learn call, or "no error"."""
    try:
        func(*args, **kwargs)
    except Exception as error:  # noqa: BLE001 - we want the name of whatever is raised
        return type(error).__name__
    return "no error"


cm_23 = skm.confusion_matrix(["a", "b", "b", "a", "b"], ["a", "b", "a", "a", "a"], normalize="pred")
sums_23 = "colonnes" if np.allclose(cm_23.sum(axis=0), 1) else "lignes" if np.allclose(cm_23.sum(axis=1), 1) else "tout"
compare_23("a", answer_23a, skm.precision_score([1, 1, 0], [0, 0, 0], zero_division=1), "zero_division")
compare_23("b", answer_23b, skm.confusion_matrix(["a", "b", "c", "a"], ["a", "c", "b", "a"], labels=["a", "b"]).tolist(), "labels")
compare_23("c", answer_23c, raised_23(skm.recall_score, ["spam", "ham", "spam"], ["spam", "spam", "ham"]), "pos_label")
compare_23("d", answer_23d, sums_23, "normalize")
compare_23("e", answer_23e, skm.accuracy_score([1, 0, 1], [1, 1, 1], sample_weight=[1, 1, 2]), "sample_weight")
compare_23("f", answer_23f, skm.precision_score(["b", "a", "c", "a"], ["b", "a", "a", "c"], average=None).tolist(), "average=None")

📝 **Les phrases de la documentation** (une par question) : …

## Partie D · Au-delà du livre : courbes, moyennes, calibration, défi

Sections « au-delà du livre » de la fiche. Tu programmes la courbe ROC et son aire (3.24), les moyennes sur plusieurs classes (3.25), la courbe precision-recall et l'average precision (3.26), puis la calibration (3.28) ; tu apprends à lire ces courbes quand la classe positive est rare (3.27), et tu finis par un défi : bloquer 99 % des fraudes avec le moins de fausses alertes possible (3.29).

### Ex 3.24 — Courbe ROC et AUC 🔨 ★★★ ⏱️ 40 min
**Objectif :** programmer la courbe ROC et l'aire sous la courbe, et s'en servir pour comparer des scores.  
**Prérequis :** Ex 3.20 · fiche « Au-delà du livre (1) » (🧮 méthode des trapèzes) · **Fil rouge :** Penguins · **mylearn :** `metrics.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/metrics.py`, mêmes règles qu'en 3.15 (NumPy permis, scikit-learn non). Enregistre, puis relance la cellule de vérification.

Écris trois fonctions (lis leurs docstrings) :
- `roc_curve(y_true, y_score, pos_label=1)` → `(fpr, tpr, thresholds)`. Convertis les entrées avec `np.asarray` (une Series pandas peut avoir un index quelconque). Trie les échantillons par score décroissant (`np.argsort(-scores, kind="mergesort")`), cumule les positifs et les négatifs rencontrés (`np.cumsum`), puis ne garde qu'un point par score distinct : des scores égaux forment un seul seuil (fiche, mini-exemple). Ajoute enfin le point de départ $(0, 0)$, de seuil `np.inf` ;
- `auc(x, y)` : l'aire par la méthode des trapèzes (fiche, 🧮), que les `x` soient croissants ou décroissants ;
- `roc_auc(y_true, y_score, pos_label=1)`, qui réutilise les deux autres.

Vérifications, pour « Gentoo contre le reste » (`is_gentoo`, comme en 3.20) :
a) l'AUC de chacune des quatre mesures de `MEASURES` prise comme score, calculée par ta fonction `roc_auc` (la cellule de vérification l'appelle) ;
b) `best_measure_24` : la mesure qui sépare le mieux les Gentoo des autres, si l'on a le droit de **retourner** un score (prendre son opposé) : son nom, une chaîne ;
c) `auc_minus_depth_24` : l'AUC de l'opposé de l'épaisseur du bec, `-measured["bill_depth_mm"]` (4 décimales), calculée avec ta fonction. Quel lien avec l'AUC de l'épaisseur elle-même ?

La vérification trace les courbes ROC des quatre mesures avec ta fonction, et contrôle sur la nageoire que l'AUC est la probabilité qu'un Gentoo tiré au hasard ait une nageoire plus longue qu'un autre manchot tiré au hasard, un ex-æquo comptant pour moitié (fiche). Puis les tests de ces trois fonctions.

In [ ]:
best_measure_24 = ...      # b) a column name
auc_minus_depth_24 = ...   # c)

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="3")   # reload your saved file
with wb.attempt("3.24"):
    aucs_24 = [mylearn.metrics.roc_auc(is_gentoo, measured[measure]) for measure in MEASURES]
    wb.check("3.24a", aucs_24, computed=True)
    gentoo_fl, other_fl = flipper[is_gentoo == 1], flipper[is_gentoo == 0]
    pairs_24 = np.mean(gentoo_fl[:, None] > other_fl[None, :]) + 0.5 * np.mean(gentoo_fl[:, None] == other_fl[None, :])
    verdict("3.24", abs(aucs_24[2] - pairs_24) < 1e-9,
            "l'AUC de la nageoire est bien la part des paires (Gentoo, autre) bien rangées, ex-æquo comptés pour moitié.",
            "l'AUC de la nageoire devrait valoir la part des paires (Gentoo, autre) bien rangées : vérifie roc_curve "
            "(un point par score distinct) et auc.")
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    for measure in MEASURES:
        fpr, tpr, _ = mylearn.metrics.roc_curve(is_gentoo, measured[measure])
        ax.plot(fpr, tpr, label=measure)
    ax.plot([0, 1], [0, 1], ls="--", color="0.5", label="random")
    ax.set(xlabel="false positive rate (FPR)", ylabel="recall (TPR)", title="Gentoo against the rest: ROC curves")
    ax.legend(loc="lower right")
    plt.show()
    run_metrics_tests("test_roc_curve_ or test_auc_ or test_roc_auc_")
wb.check("3.24b", best_measure_24)
wb.check("3.24c", auc_minus_depth_24)

### Ex 3.25 — Moyennes macro, micro et pondérée 🔨 ★★★ ⏱️ 35 min
**Objectif :** étendre precision, recall, F-beta et F1 à plusieurs classes, et voir quelle moyenne remarque une classe oubliée.  
**Prérequis :** Ex 3.16, Ex 3.6, Ex 3.15 · Ex 1.15 · fiche « Au-delà du livre : plusieurs classes » · **Fil rouge :** Penguins · **mylearn :** `metrics.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/metrics.py`, mêmes règles qu'en 3.15 (NumPy permis, scikit-learn non). Enregistre, puis relance la cellule de vérification.

Complète `precision`, `recall`, `fbeta` et `f1` pour les autres valeurs d'`average` (docstrings) :
- `None` : une valeur par classe, chaque classe devenant tour à tour la classe positive (*one-vs-rest*), dans l'ordre trié des étiquettes présentes dans `y_true` **et** `y_pred`. Ta `confusion_matrix` (3.15) donne tout d'un coup : les TP sur sa diagonale, les FP dans le reste de chaque colonne, les FN dans le reste de chaque ligne ;
- `"macro"` : la moyenne simple des valeurs par classe ; `"weighted"` : leur moyenne pondérée par le support, le nombre de vrais échantillons de chaque classe (`np.average(..., weights=...)`) ;
- `"micro"` : additionne d'abord les TP, les FP et les FN de toutes les classes, puis calcule **une seule** mesure ;
- toute autre valeur d'`average` lève une `ValueError`.

Une classe jamais prédite a une precision 0/0 : c'est `zero_division` qui décide.

Deux classifieurs à trois classes sur les 342 manchots : les règles de la biologiste (`expert_pred`) et une règle à **deux** espèces, `two_species_pred` : « Gentoo » si la nageoire mesure au moins 210 mm, « Adelie » sinon ; elle n'annonce donc jamais « Chinstrap ». Vérifications (la cellule de vérification appelle tes fonctions) :
a) le F1 de chaque espèce pour la règle à deux espèces (`average=None`) ;
b) `[macro, micro, weighted]` du F1 des règles de la biologiste ;
c) la même liste pour la règle à deux espèces ;
d) `most_sensitive_25` : la moyenne qui baisse le plus d'un classifieur à l'autre : `"macro"`, `"micro"` ou `"weighted"`.

La vérification contrôle aussi que le F1 micro vaut l'accuracy (fiche), puis lance les tests des moyennes. Dans tes notes : quand préférer la moyenne macro ? (fiche, E4)

In [ ]:
two_species_pred = np.where(flipper >= 210, "Gentoo", "Adelie")   # never answers "Chinstrap"

In [ ]:
most_sensitive_25 = ...   # d) "macro", "micro" or "weighted"

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="3")   # reload your saved file
with wb.attempt("3.25"):
    mm = mylearn.metrics
    wb.check("3.25a", mm.f1(species, two_species_pred, average=None), computed=True)
    wb.check("3.25b", [mm.f1(species, expert_pred, average=average) for average in ("macro", "micro", "weighted")], computed=True)
    wb.check("3.25c", [mm.f1(species, two_species_pred, average=average) for average in ("macro", "micro", "weighted")], computed=True)
    verdict("3.25", abs(mm.f1(species, expert_pred, average="micro") - np.mean(species == expert_pred)) < 1e-12,
            "le F1 micro vaut l'accuracy : chaque erreur est à la fois un FP (pour la classe prédite) et un FN (pour la vraie).",
            "le F1 micro devrait valoir l'accuracy : additionne les TP, FP et FN de toutes les classes AVANT de calculer.")
    run_metrics_tests("(test_precision_ or test_recall_ or test_fbeta_ or test_f1_) and multiclass and not curve")
wb.check("3.25d", most_sensitive_25)

### Ex 3.26 — Courbe precision-recall et average precision 🔨 ★★★ ⏱️ 40 min
**Objectif :** programmer la courbe precision-recall et l'average precision, puis comparer deux modèles selon le nombre d'alertes qu'on peut traiter.  
**Prérequis :** Ex 3.24 · fiche « Au-delà du livre (2) » · **Fil rouge :** synthétique · **mylearn :** `metrics.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/metrics.py`, mêmes règles qu'en 3.15 (NumPy permis, scikit-learn non). Enregistre, puis relance la cellule de vérification.

Écris `precision_recall_curve(y_true, y_score, pos_label=1)` et `average_precision(y_true, y_score, pos_label=1)` (docstrings). Tu peux reprendre le cœur de ta `roc_curve` (tri par score décroissant, cumuls, un point par score distinct) : une fonction d'aide commune évite de l'écrire deux fois. Attention aux conventions de scikit-learn : les seuils sont rangés par ordre **croissant**, et un dernier point (precision 1, recall 0), sans seuil, ferme la courbe. L'AP est une somme **en escalier** (fiche), pas des trapèzes.

Une équipe antifraude compare deux modèles, A et B, sur 5 000 transactions dont environ 4 % de fraudes (`y_26`, `score_a`, `score_b`).
a) `[AP de A, AP de B]`, calculées par ta fonction `average_precision` (la cellule de vérification l'appelle) ;
b) `top50_26` : la precision de chaque modèle parmi ses 50 transactions les plus suspectes, c'est-à-dire ses 50 plus hauts scores (liste `[A, B]`, 2 décimales) ;
c) `top300_26` : la même chose parmi ses 300 plus hauts scores (3 décimales) ;
d) `choice_26` : l'équipe peut vérifier 300 alertes par jour : quel modèle choisir, `"A"` ou `"B"` ?
e) `auc_26` : `[AUC de A, AUC de B]` (2 décimales), avec ta fonction `roc_auc` de 3.24.

La vérification trace les deux courbes precision-recall avec ta fonction. Dans tes notes : l'AP, l'AUC et le choix de d désignent-ils le même modèle ? Que « voit » chacune de ces mesures ?

In [ ]:
rng_26 = np.random.default_rng(26)
y_26 = (rng_26.random(5000) < 0.04).astype(int)           # 1 = fraud, 0 = normal transaction
easy_26 = rng_26.random(5000) < 0.4                       # the frauds that model A recognises at once
score_a = np.where((y_26 == 1) & easy_26, rng_26.normal(4, 0.5, 5000), rng_26.normal(0, 1, 5000) + 0.8 * y_26)
score_b = rng_26.normal(0, 1, 5000) + 2.0 * y_26          # model B: every fraud a little higher
print(f"{y_26.sum()} frauds among {len(y_26)} transactions")

In [ ]:
top50_26 = ...    # b) [A, B]
top300_26 = ...   # c) [A, B]
choice_26 = ...   # d) "A" or "B"
auc_26 = ...      # e) [A, B]

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="3")   # reload your saved file
with wb.attempt("3.26"):
    wb.check("3.26a", [mylearn.metrics.average_precision(y_26, score) for score in (score_a, score_b)], computed=True)
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for name, score in [("A", score_a), ("B", score_b)]:
        precision, recall, _ = mylearn.metrics.precision_recall_curve(y_26, score)
        ax.plot(recall, precision, label=f"model {name}")
    ax.set(xlabel="recall", ylabel="precision", title="Two fraud models: precision-recall curves", ylim=(0, 1.02))
    ax.legend()
    plt.show()
    run_metrics_tests("test_precision_recall_curve_ or test_average_precision_")
wb.check("3.26b", top50_26)
wb.check("3.26c", top300_26)
wb.check("3.26d", choice_26)
wb.check("3.26e", auc_26)

### Ex 3.27 — ROC ou PR ? Lire les courbes d'un problème déséquilibré 📈 ★★ ⏱️ 25 min
**Objectif :** lire des courbes ROC et precision-recall quand la classe positive est rare, et traduire un taux de faux positifs en nombre de fausses alertes.  
**Prérequis :** Ex 3.24 · fiche « Au-delà du livre (1) et (2) » (🕰️ PR et AP) · **Fil rouge :** synthétique · **Parcours :** R, M

Un même modèle (les mêmes lois de scores pour les positifs et pour les négatifs) est évalué sur deux populations de 200 000 cas : l'une compte 2 % de positifs, l'autre 0,5 %. La cellule ci-dessous trace, avec scikit-learn, les courbes ROC (à gauche, puis un zoom sur les petits taux de faux positifs) et precision-recall (à droite). Lis les graphiques, sans recalculer :

a) `same_roc_27` : les deux courbes ROC sont-elles presque confondues ? (`True` ou `False`)
b) `fpr_at_half_27` : sur le zoom, le taux de faux positifs quand le recall (TPR) vaut 0,5 (2 décimales) ;
c) `precision_at_half_27` : sur les courbes precision-recall, la precision quand le recall vaut 0,5, pour 2 % puis pour 0,5 % de positifs (liste, 1 décimale) ;
d) `baseline_27` : la precision d'un classifieur qui répond au hasard, avec 0,5 % de positifs (3 décimales) : à quelle hauteur serait sa « courbe » PR ?
e) `curve_27` : laquelle des deux courbes, `"ROC"` ou `"PR"`, montre que la plupart des alertes seraient fausses avec 0,5 % de positifs ?

Dans tes notes, relie b et c : avec 0,5 % de positifs, combien y a-t-il de négatifs parmi les 200 000 cas, donc combien de fausses alertes au taux lu en b, pour combien de vrais positifs trouvés au recall 0,5 ? Retrouves-tu la precision lue en c ?

In [ ]:
rng_27 = np.random.default_rng(27)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for share, colour in [(0.02, "C0"), (0.005, "C1")]:
    y = rng_27.random(200_000) < share                     # the positives of this population
    score = rng_27.normal(0, 1, 200_000) + 2.0 * y         # the same laws of scores in both populations
    fpr, tpr, _ = skm.roc_curve(y, score)
    precision, recall, _ = skm.precision_recall_curve(y, score)
    label = f"{share:.1%} of positives"
    axes[0].plot(fpr, tpr, color=colour, label=label)
    axes[1].plot(fpr, tpr, color=colour, label=label)
    axes[2].plot(recall, precision, color=colour, label=label)
axes[0].plot([0, 1], [0, 1], ls="--", color="0.5", label="random")
axes[0].set(title="ROC", xlabel="false positive rate (FPR)", ylabel="recall (TPR)")
axes[1].set(title="ROC, zoom on the small FPR", xlabel="false positive rate (FPR)", ylabel="recall (TPR)",
            xlim=(0, 0.05), xticks=np.arange(0, 0.051, 0.01), ylim=(0, 1.02))
axes[1].axhline(0.5, color="0.5", ls=":")
axes[2].set(title="precision-recall", xlabel="recall", ylabel="precision", ylim=(0, 1.02), yticks=np.arange(0, 1.01, 0.1))
axes[2].axvline(0.5, color="0.5", ls=":")
for ax in axes:
    ax.grid(True, alpha=0.4)
    ax.legend(loc="lower right" if ax is not axes[2] else "upper right")
plt.show()

In [ ]:
same_roc_27 = ...            # a) True or False
fpr_at_half_27 = ...         # b)
precision_at_half_27 = ...   # c) [2 %, 0.5 %]
baseline_27 = ...            # d)
curve_27 = ...               # e) "ROC" or "PR"

In [ ]:
for letter, answer in zip("abcde", [same_roc_27, fpr_at_half_27, precision_at_half_27, baseline_27, curve_27]):
    wb.check(f"3.27{letter}", answer)

### Ex 3.28 — Calibration : quand la météo annonce 70 % 🔨 ★★★ ⏱️ 35 min
**Objectif :** programmer le diagramme de fiabilité et le score de Brier, et distinguer calibration et qualité du classement.  
**Prérequis :** Ex 3.16, Ex 2.21 · fiche §3.1 et « Au-delà du livre (3) » · **Fil rouge :** synthétique · **mylearn :** `metrics.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/metrics.py`, mêmes règles qu'en 3.15 (NumPy permis, scikit-learn non). Enregistre, puis relance la cellule de vérification.

Écris `calibration_curve(y_true, y_prob, n_bins=10)` et `brier_score(y_true, y_prob)` (docstrings). Pour trouver l'intervalle de chaque probabilité, **compare-la aux bords** `edges = np.linspace(0, 1, n_bins + 1)` : `np.searchsorted(edges[1:-1], p)` (ou `np.digitize(p, edges[1:-1], right=True)`) donne le numéro de l'intervalle, une valeur posée sur un bord intérieur allant dans l'intervalle du bas. Ne calcule pas ce numéro par `int(p * n_bins)` : une probabilité posée sur un bord intérieur (0,4 avec 10 intervalles) irait dans l'intervalle du haut au lieu de celui du bas, et $p = 1$ tomberait hors des intervalles ; un test le vérifie. `np.bincount(..., weights=..., minlength=n_bins)` fait ensuite les sommes par intervalle ; ne garde que les intervalles non vides.

Dix ans de prévisions de pluie (3 650 jours) : `rain` vaut 1 les jours de pluie, 0 sinon. Trois prévisionnistes annoncent chaque matin une probabilité de pluie, fabriquée par la cellule ci-dessous : A annonce la vraie probabilité du jour, B la pousse vers 0 ou vers 1, C annonce tous les jours la même valeur, la probabilité moyenne de pluie.
a) `[Brier de A, Brier de B, Brier de C]`, calculés par ta fonction `brier_score` (la cellule de vérification l'appelle) ;
b) `shows_70_28` : une application météo affiche les probabilités arrondies à 10 % : elle affiche « 70 % » quand la prévision est entre 0,65 (inclus) et 0,75 (exclu). Ces jours-là, quelle est la fréquence de la pluie, pour A puis pour B (liste, 3 décimales) ?
c) `auc_28` : `[AUC de A, AUC de B, AUC de C]` (3 décimales), avec ta fonction `roc_auc` de 3.24 ou `skm.roc_auc_score`.

La vérification trace le diagramme de fiabilité des trois prévisionnistes avec ta fonction. Dans tes notes : lequel est calibré ? Lequel est trop sûr de lui, et où le voit-on sur le diagramme ? Lequel est calibré mais inutile, et pourquoi ?

In [ ]:
rng_28 = np.random.default_rng(28)
chance_28 = rng_28.beta(1.2, 2.0, 3650)                  # the true chance of rain of each day (known to nobody)
rain = (rng_28.random(3650) < chance_28).astype(int)     # 1 = it rained that day
forecast_a = chance_28                                   # A announces the true chance: honest
forecast_b = 1 / (1 + np.exp(-2.5 * np.log(chance_28 / (1 - chance_28))))   # B pushes it towards 0 or 1
forecast_c = np.full(3650, 0.375)                        # C always announces the average chance of rain
print(f"{rain.sum()} rainy days out of {len(rain)}")

In [ ]:
shows_70_28 = ...   # b) [A, B]
auc_28 = ...        # c) [A, B, C]

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="3")   # reload your saved file
with wb.attempt("3.28"):
    wb.check("3.28a", [mylearn.metrics.brier_score(rain, forecast) for forecast in (forecast_a, forecast_b, forecast_c)], computed=True)
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.plot([0, 1], [0, 1], ls="--", color="0.5", label="perfectly calibrated")
    for name, forecast in [("A", forecast_a), ("B", forecast_b), ("C", forecast_c)]:
        prob_true, prob_pred = mylearn.metrics.calibration_curve(rain, forecast, n_bins=10)
        ax.plot(prob_pred, prob_true, marker="o", label=f"forecaster {name}")
    ax.set(xlabel="announced probability (mean of each bin)", ylabel="observed frequency of rain",
           title="Reliability diagram (10 bins)", xlim=(0, 1), ylim=(0, 1))
    ax.legend()
    plt.show()
    run_metrics_tests("test_calibration_curve_ or test_brier_score_")
wb.check("3.28b", shows_70_28)
wb.check("3.28c", auc_28)

### Ex 3.29 — Recall ≥ 0,99 au meilleur prix 🏆 ★★★ ⏱️ 45 min
**Objectif :** choisir, sur un jeu de validation seulement, un modèle et un seuil qui bloquent 99 % des fraudes du jeu de test avec le moins de fausses alertes possible.  
**Prérequis :** Ex 3.26, Ex 3.20 · ch. 2 (percentiles, bootstrap) · fiche §3.7.4, « Au-delà du livre (2) », pièges · **Fil rouge :** synthétique · **Parcours :** C

**Défi.** Une banque exige que son système bloque **au moins 99 %** des fraudes (recall ≥ 0,99). Chaque alerte coûte un appel au client : elle veut aussi **au moins une vraie fraude pour 10 alertes** (precision ≥ 0,10).

Deux modèles, A et B, donnent un score à chaque transaction (1 % de fraudes). Tu disposes d'un jeu de **validation** (`val_29` : 50 000 transactions, colonnes `score_a`, `score_b` et `fraud`) pour faire tes choix, et d'un jeu de **test** (`test_29` : 50 000 autres transactions) qui ne sert qu'à la note finale. Ne le regarde pas pour choisir : régler un seuil sur le test, c'est tricher (fiche, pièges).

Écris `choose_29(val)`, qui reçoit un DataFrame de validation et renvoie `(column, threshold)` : la colonne de score retenue (`"score_a"` ou `"score_b"`) et le seuil (une alerte quand score ≥ seuil). La vérification applique ton choix au jeu de test : il faut un recall ≥ 0,99 **et** une precision ≥ 0,10.

Deux pièges t'attendent :
1. le meilleur modèle selon l'AP n'est pas forcément le meilleur quand il faut attraper presque **toutes** les fraudes : regarde les scores des fraudes les plus basses de chaque modèle ;
2. un seuil qui donne exactement 99 % de recall sur la validation en donne souvent moins sur le test : les fraudes du test ne sont pas celles de la validation, et le 1 % le plus bas d'environ 500 fraudes varie beaucoup d'un tirage à l'autre. Prends une **marge**, mais pas trop grande, sinon la precision s'effondre. Le bootstrap du ch. 2 peut t'aider à mesurer cette variabilité.

Pour aller plus loin, la vérification rejoue ta méthode sur 20 autres couples (validation, test) : combien de fois tient-elle ses deux promesses ?

In [ ]:
def make_transactions(n, seed):
    """n transactions, about 1 % of frauds, with the scores of two models A and B."""
    rng = np.random.default_rng(seed)
    fraud = rng.random(n) < 0.01
    kind_2 = rng.random(n) < 0.06
    score_a = rng.normal(0, 1, n) + np.where(fraud & ~kind_2, 5.0, 0.0)
    score_b = rng.normal(0, 1, n) + 4.0 * fraud
    return pd.DataFrame({"score_a": score_a, "score_b": score_b, "fraud": fraud.astype(int)})


def evaluate_29(column, threshold, data):
    """(recall, precision, number of alerts) when we raise an alert for data[column] >= threshold."""
    alert = data[column].to_numpy() >= threshold
    fraud = data["fraud"].to_numpy() == 1
    caught = int(np.sum(alert & fraud))
    return caught / int(np.sum(fraud)), caught / max(int(np.sum(alert)), 1), int(np.sum(alert))


val_29 = make_transactions(50_000, seed=291)     # to choose the model and the threshold
test_29 = make_transactions(50_000, seed=391)    # only for the final grade: do not look at it to choose
print(f"validation: {val_29['fraud'].sum()} frauds · test: {test_29['fraud'].sum()} frauds (50 000 transactions each)")

In [ ]:
def choose_29(val):
    """(score column, threshold) chosen with the validation DataFrame only."""
    raise NotImplementedError("choose_29() is not written yet")

In [ ]:
with wb.attempt("3.29"):
    column_29, threshold_29 = choose_29(val_29)
    recall_29, precision_29, alerts_29 = evaluate_29(column_29, threshold_29, test_29)
    print(f"your choice: {column_29}, threshold {threshold_29:.4f} · test set: {alerts_29} alerts, "
          f"recall {recall_29:.4f}, precision {precision_29:.4f}")
    verdict("3.29", recall_29 >= 0.99, "recall ≥ 0,99 sur le test : au moins 99 % des fraudes sont bloquées.",
            "recall < 0,99 sur le test : trop de fraudes passent ; prends plus de marge (ou vérifie le modèle).")
    verdict("3.29", precision_29 >= 0.10, "precision ≥ 0,10 sur le test : au plus 9 fausses alertes par fraude bloquée.",
            "precision < 0,10 sur le test : trop d'alertes ; ta marge est trop grande, ou le modèle n'est pas le bon.")
    held_29 = [evaluate_29(*choose_29(make_transactions(50_000, 1000 + 2 * k)), make_transactions(50_000, 1001 + 2 * k))
               for k in range(20)]
    print(f"for information, your method on 20 other (validation, test) pairs: recall ≥ 0.99 in "
          f"{sum(r >= 0.99 for r, _, _ in held_29)}, precision ≥ 0.10 in {sum(p >= 0.10 for _, p, _ in held_29)}, "
          f"both in {sum(r >= 0.99 and p >= 0.10 for r, p, _ in held_29)}")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une probabilité conditionnelle dans une table, et dire laquelle de $P(A \mid B)$ ou $P(B \mid A)$ répond à une question ?
2. Sais-tu construire une matrice de confusion, en tirer les mesures, et choisir une mesure et un seuil selon le coût des erreurs ?
3. Sais-tu expliquer pourquoi, quand la maladie est rare, la plupart des résultats positifs d'un très bon test peuvent être faux, et quand montrer une courbe precision-recall plutôt qu'une ROC ?

**Pour aller plus loin** : les pages du *Machine Learning Crash Course* de Google et le guide « Metrics and scoring » de scikit-learn, cités dans la fiche ; l'article de Fawcett (3.11). La suite : le ch. 4 (règle de Bayes), qui passe de $P(\text{positif} \mid \text{malade})$ à $P(\text{malade} \mid \text{positif})$ ; `mylearn.metrics` servira dans tous les chapitres suivants, dès le ch. 7 (classification) et le ch. 8 (choisir un seuil sur un jeu de validation).